# RSNA Knee: what the top public notebooks are actually made of

There are dozens of public notebooks at 0.950 and above, with different titles, authors and blend weights.
Under the hood they draw from a handful of public model datasets. This notebook reads, live from Kaggle,
which datasets every top notebook attaches, and answers three practical questions:

1. **Which public models does the top of the leaderboard actually rest on?**
2. **Which recipes (combinations of model datasets) reach which score?**
3. **Which top notebooks depend on an input you cannot load?** A copy of those runs without that input and
   scores lower than the author did.

Nothing here touches competition data: it reads the same public notebook pages you see in the browser.
Turn **Internet on** to run it; a fork always shows today's state.

In [ ]:
import re
import time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import pandas as pd
import requests

COMPETITION = "rsna-knee-abnormality-detection"
MIN_SCORE = 0.950
pd.set_option("display.max_colwidth", 80)

_s = requests.Session()
_s.get("https://www.kaggle.com/competitions", timeout=60)
_h = {"x-xsrf-token": _s.cookies.get("XSRF-TOKEN", "")}


def kaggle_api(service, method, body, tries=4):
    for attempt in range(tries):
        r = _s.post(f"https://www.kaggle.com/api/i/{service}/{method}", json=body, headers=_h, timeout=90)
        if r.status_code == 200:
            return r.json()
        time.sleep(3 * (attempt + 1))
    raise RuntimeError(f"{method}: HTTP {r.status_code}. Is Internet turned on?")


comp = kaggle_api("competitions.CompetitionService", "GetCompetition", {"competitionName": COMPETITION})
comp = comp.get("competition", comp)
notebooks = []
for page in range(1, 6):
    body = {"kernelFilterCriteria": {"search": "", "listRequest": {
                "competitionId": comp["id"], "sortBy": "SCORE_DESCENDING", "pageSize": 100, "group": "EVERYONE", "page": page}},
            "detailFilterCriteria": {"deletedAccessBehavior": "RETURN_NOTHING", "unauthorizedAccessBehavior": "RETURN_NOTHING",
                                     "excludeResultsFilesOutputs": True},
            "readMask": "pinnedKernels"}
    got = kaggle_api("kernels.KernelsService", "ListKernels", body).get("kernels", [])
    notebooks += got
    if not got or (got[-1].get("bestPublicScore") or 0) < MIN_SCORE:
        break

rows = []
for k in notebooks:
    score = k.get("bestPublicScore")
    if score is None or score < MIN_SCORE:
        continue
    sources = [d.get("dataSourceUrl", "") for d in (k.get("dataSources") or [])]
    models = sorted(s.split("/", 2)[2] for s in sources if s.startswith(("/datasets/", "/models/", "/code/")))
    rows.append({"notebook": k["scriptUrl"].removeprefix("/code/"), "score": score, "votes": k.get("totalVotes") or 0,
                 "updated": str(k.get("scriptVersionDateCreated", ""))[:10], "inputs": models,
                 "hidden inputs": sum(1 for s in sources if not s)})
top = pd.DataFrame(rows).sort_values(["score", "votes"], ascending=False).reset_index(drop=True)
print(f"{len(top)} public notebooks score {MIN_SCORE} or more (of {len(notebooks)} fetched)")

## 1. The handful of public models under the top of the leaderboard

How many of the top notebooks attach each dataset, and the best score reached by a notebook that uses it.
`(hidden)` counts inputs that do not appear on the public page: private datasets a copy cannot load.

In [ ]:
KNOWN = {  # what is inside the main public model datasets (from their own pages and notebooks)
    "nartaa/rsna-knee-publication-swa-weights-20261007": "two CoAtNet checkpoints: 384 px (0.949 alone) and 224 px / 0.8 crop (0.945 alone)",
    "heliosli/rsna-orthofoundation-fold0-epoch7-20261004": "heliosli's inference code + anatomy model used by their ConvNeXt stages",
    "heliosli/rsna-knee-compact-v1": "heliosli's ConvNeXt-small 'S6', four folds",
    "heliosli/rsna-knee-compact-r6cnxfsss": "same S6 files as compact-v1 (owner-only for submissions)",
    "heliosli/rsna-knee-compact-r7cnxfsss": "heliosli's ConvNeXt-small 'R7': crop 352, fastMRI auxiliary training",
    "goodpjw2008/rsna-knee-2-5d-convnext-reader": "goodpjw2008's 2.5D ConvNeXt-Tiny reader, three folds (0.929 alone)",
}
use, best = Counter(), defaultdict(float)
for inputs, score in zip(top["inputs"], top["score"]):
    for d in set(inputs):
        use[d] += 1
        best[d] = max(best[d], score)
hidden = int((top["hidden inputs"] > 0).sum())
usage = pd.DataFrame([{"dataset": d, "top notebooks using it": n, "best score": best[d], "what it is": KNOWN.get(d, "")}
                      for d, n in use.most_common() if not d.startswith("competitions/")])
usage = pd.concat([usage, pd.DataFrame([{"dataset": "(hidden / private input)", "top notebooks using it": hidden,
                                         "best score": top.loc[top["hidden inputs"] > 0, "score"].max(), "what it is": "cannot be loaded by a copy"}])])
shown = usage.head(12).iloc[::-1]

INK, INK2, MUTED, GRID, BLUE, ORANGE, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#2a78d6", "#eb6834", "#fcfcfb"
fig, ax = plt.subplots(figsize=(11, 0.42 * len(shown) + 1.6), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
colors = [ORANGE if d.startswith("(hidden") else BLUE for d in shown["dataset"]]
ax.barh(shown["dataset"].str.slice(0, 55), shown["top notebooks using it"], color=colors, height=0.65)
for y, (n, s) in enumerate(zip(shown["top notebooks using it"], shown["best score"])):
    ax.text(n + 0.4, y, f"{n}  (best {s:.3f})", va="center", fontsize=9.5, color=INK2)
ax.set_xlabel(f"number of public notebooks scoring {MIN_SCORE}+ that attach it", color=INK2)
ax.set_title(f"What the {len(top)} top public notebooks attach", loc="left", fontsize=13, color=INK, fontweight="bold")
ax.grid(axis="x", color=GRID, linewidth=0.8); ax.set_axisbelow(True)
for s in ("top", "right"): ax.spines[s].set_visible(False)
for s in ("left", "bottom"): ax.spines[s].set_color(MUTED)
ax.tick_params(colors=INK2)
ax.set_xlim(0, shown["top notebooks using it"].max() * 1.3)
plt.tight_layout(); plt.show()
usage.reset_index(drop=True)

## 2. Recipes: which combination of models reaches which score

A *recipe* here is the set of model datasets a notebook attaches. Notebooks with the same recipe differ
only in blend weights and small code changes, which is why so many of them land on the same displayed
score.

In [ ]:
def label(inputs):
    short = []
    for d in inputs:
        if d.startswith("competitions/"):
            continue
        name = d.split("/")[-1] if "/" in d else d
        short.append(name.replace("rsna-knee-", "").replace("-20261007", "").replace("-20261004", "")[:28])
    return " + ".join(sorted(short)) or "(competition data only)"


top["recipe"] = top["inputs"].map(label) + top["hidden inputs"].map(lambda h: " + (hidden)" if h else "")
recipes = (top.groupby("recipe")
           .agg(notebooks=("score", "size"), best=("score", "max"), median=("score", "median"), votes=("votes", "sum"))
           .sort_values(["best", "notebooks"], ascending=False).reset_index())
recipes

## 3. Top notebooks a copy cannot reproduce

These attach at least one input that is not visible on their public page, usually the author's own private
model. A copy runs without it, so it scores what the public part scores, not what the title says.

In [ ]:
top.loc[top["hidden inputs"] > 0, ["notebook", "score", "votes", "updated", "hidden inputs"]].reset_index(drop=True)

## What this means if you are trying to climb

- **The top is a monoculture.** Nearly every high-scoring public notebook combines the same few model
  datasets. Copying another one rarely adds anything: same models, different weights.
- **Diversity is the lever.** A model that is wrong on *different* studies helps a blend more than a
  slightly stronger copy of what is already there. One concrete case: nartaa's dataset ships a second,
  224-pixel checkpoint that almost no public blend uses; adding it moved my rank inside the 0.956 block
  without changing the displayed score
  ([notebook](https://www.kaggle.com/code/varshithakolupuri/rsna-knee-one-unused-checkpoint-lifts-the-0-956)).
- **Check section 3 before forking.** A high score built on a hidden input will not reproduce.

Related: [real scores of every public notebook and the leaderboard's tie blocks](https://www.kaggle.com/code/varshithakolupuri/rsna-knee-real-scores-of-every-public-notebook).

Credit to every author whose public models are listed here: the leaderboard rests on their work. If this
map saved you a wasted fork, an upvote helps others find it.